Run the next cell first. In Colab, add the course key under the key
icon in the left sidebar as `COURSE_API_KEY`, with *Notebook access* on.

In [ ]:
# Course setup. In Colab: add the course key under the key icon (left sidebar)
# as COURSE_API_KEY. On your own machine it uses Ollama instead.
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Pinned, including the transitive ones. Unpinned, pip takes whatever
    # shipped this morning : a newer core moves ModelError, and a newer openai
    # rejects this endpoint's usage payload. Keep in step with requirements.txt.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "langchain==1.3.9", "langchain-core==1.4.7",
                    "langchain-openai==1.3.2", "langchain-classic==1.0.8",
                    "langgraph==1.2.5", "openai==2.41.1", "python-dotenv"],
                   check=True)
    MODEL = "qwen3.8-flash"          # try qwen3.8-max too
    BASE = "https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1"
    THINKING = {"enable_thinking": False}
    KEY = os.getenv("COURSE_API_KEY")
    if not KEY:
        try:
            from google.colab import userdata
            KEY = userdata.get("COURSE_API_KEY")   # raises if unset or not shared
        except Exception:
            import getpass
            KEY = getpass.getpass("Course key (paste the one from the trainer): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    MODEL = "qwen3.5:2b"             # try qwen3.5:4b too
    BASE = "http://localhost:11434/v1"
    KEY = "ollama"
    THINKING = {"reasoning_effort": "none"}

from langchain_openai import ChatOpenAI


def make_llm(**kw):
    kw.setdefault("temperature", 0)
    kw.setdefault("model", MODEL)
    kw.setdefault("extra_body", THINKING)
    return ChatOpenAI(base_url=BASE, api_key=KEY, **kw)


def make_embeddings(**kw):
    # The course endpoint has no embeddings, so they run here instead. 90 MB,
    # installed only by the notebooks that actually ask for them.
    try:
        from langchain_huggingface import HuggingFaceEmbeddings
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "langchain-huggingface==0.3.1"], check=True)
        from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


DATA_URL = ("https://raw.githubusercontent.com/FeikoWielsma/"
            "Building-AI-Agents/main/data/")


def data(name):
    """Path to a course data file. Downloads it in Colab, local copy otherwise."""
    if os.path.exists(f"../data/{name}"):
        return f"../data/{name}"
    if not os.path.exists(name):
        import urllib.request
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name


llm = make_llm()
print(f"model={MODEL}")

# Solution - Router Agent Pattern in LangGraph

A router agent looks at an incoming request, decides which specialist should handle it, and dispatches to exactly one of several branches. Unlike the sequential pattern (fixed order), the router uses a conditional edge to pick a destination:

```
                        -> [math]     ->
START -> [classify] --->  -> [translate] --> END
                        -> [general]  ->
```

The `classify` node is the router's 'brain' - it reads the question and writes a `route` value into the state. A conditional edge then sends the state to the matching specialist, which produces the answer.

## 0. Install & imports

In [ ]:
# (setup cell already installs what this notebook needs)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

# LLM on the local Ollama bridge (temperature 0 for a stable classifier)
llm = make_llm()

## 1. Define the shared state

The router needs a `route` field (the classifier's decision) on top of `question` and `answer`.

In [ ]:
class RouterState(TypedDict, total=False):
    question: str   # input:  the user's request
    route: str      # set by the classifier: "math" | "translate" | "general"
    answer: str     # set by the chosen specialist

## 2. Define the classifier and the specialists

`classify` decides the route; each specialist answers with its own prompt. One model, different jobs per branch.

In [ ]:
def classify(state: RouterState) -> RouterState:
    """Router 'brain': ask the LLM to pick a category, then normalise it to one
    of the three route names (defaulting to 'general' if the model is vague)."""
    question = state["question"]
    prompt = (
        "Classify the user's request into exactly one word: "
        "'math' (calculations), 'translate' (translation requests), or "
        "'general' (anything else). Respond with only that one word.\n\n"
        f"Request: {question}"
    )
    raw = llm.invoke(prompt).content.strip().lower()
    if "math" in raw:
        state["route"] = "math"
    elif "translat" in raw:
        state["route"] = "translate"
    else:
        state["route"] = "general"
    return state


def math_specialist(state: RouterState) -> RouterState:
    prompt = f"You are a math tutor. Solve this and show the result:\n{state['question']}"
    state["answer"] = llm.invoke(prompt).content
    return state


def translate_specialist(state: RouterState) -> RouterState:
    prompt = f"You are a translator. Carry out this translation request:\n{state['question']}"
    state["answer"] = llm.invoke(prompt).content
    return state


def general_specialist(state: RouterState) -> RouterState:
    prompt = f"You are a helpful assistant. Answer concisely:\n{state['question']}"
    state["answer"] = llm.invoke(prompt).content
    return state

## 3. Build the graph

The defining piece of the router pattern is `add_conditional_edges`: it runs the `route` function after `classify` and uses the returned string to choose the next node.

In [ ]:
builder = StateGraph(RouterState)
builder.add_node("classify", classify)
builder.add_node("math", math_specialist)
builder.add_node("translate", translate_specialist)
builder.add_node("general", general_specialist)

# Always start by classifying
builder.add_edge(START, "classify")


# The router reads the classifier's decision and returns the next node name
def route(state: RouterState) -> str:
    return state["route"]


# Conditional edge: dispatch from "classify" to the matching specialist
builder.add_conditional_edges(
    "classify",
    route,
    {"math": "math", "translate": "translate", "general": "general"},
)

# Every specialist ends the run
builder.add_edge("math", END)
builder.add_edge("translate", END)
builder.add_edge("general", END)

graph = builder.compile()

## 4. Run the router

Each question should be classified and sent to a different specialist. The `route` field shows which branch was taken.

In [ ]:
for q in [
    "What is 15 multiplied by 7?",
    "Translate 'good morning' into Dutch.",
    "Who wrote Hamlet?",
]:
    result = graph.invoke({"question": q})
    print(f"Q: {q}")
    print(f"   route  -> {result['route']}")
    print(f"   answer -> {result['answer']}\n")

## 5. Bonus - which branch fired?

`graph.stream(...)` yields after each node, so the list of nodes run reveals the path taken: `classify` then exactly one specialist.

In [ ]:
# Bonus: see which specialist actually fired for each question
for q in ["Add 8 and 5", "Translate 'thank you' into French", "Name a planet"]:
    fired = [list(step.keys())[0] for step in graph.stream({"question": q})]
    print(f"{q!r:45} -> nodes run: {fired}")

### Extension ideas

- Add a 4th specialist (e.g. `code` for programming questions) - note it's three edits: a node, a route value in `classify`, and an entry in the conditional-edge mapping.
- Replace the string-parsing classifier with structured output (`response_format` / a Pydantic label) for a more robust route.
- Chain a router *into* a sequential pipeline: route first, then run the chosen specialist through a multi-stage sequence.